In [1]:
%pip install pandas numpy scipy scikit-learn joblib openpyxl

Note: you may need to restart the kernel to use updated packages.


# Backward Elimination — 7 Paradigmas de Aprendizado de Máquina

Para cada arquivo `Ranking_*.xlsx` gerado no notebook de ranking, este notebook parte da lista completa de atributos (na ordem do ranking, do melhor para o pior) e remove progressivamente o atributo pior colocado, avaliando a cada passo um modelo com validação cruzada estratificada (5-fold).

**Saída consolidada**: o notebook grava **1 arquivo por paradigma** (`Modelos<Paradigma>_CV_Consolidado.xlsx`), em que **cada aba é um ranking**. A aba usa o nome do ranking (ex.: `Ranking_ApproxMIC`), respeitando o limite de 31 caracteres e os caracteres proibidos do Excel (`[ ] : * ? / \`); veja a nota na seção 5 sobre o nome completo, registrado em `Mapa_Abas_Consolidado.xlsx`.

**Funcionalidades adicionais**:

- **Valores por fold**: cada linha do Excel traz também os **5 valores brutos por fold** de cada métrica (coluna `{Métrica}_Por_Fold`), pré-requisito para testes pareados (Wilcoxon).
- **Redundância de variável** (seção 6): compara, para cada ranking e paradigma, o desempenho antes e depois de cada remoção do Backward Elimination. Se a acurácia balanceada não piora de forma estatisticamente significativa (Wilcoxon pareado nos 5 folds), a variável removida naquele passo é marcada como **redundante**.
- **Sétimo paradigma**: Regressão Logística com regularização L1 (Lasso).
- **Alerta de AUC**: sinaliza quando `AUC Média` fica abaixo de 0,50 (pior que aleatório).
- **Teste de Wilcoxon pareado**: a função `teste_wilcoxon_pareado` é usada na seção 6; a comparação entre a proposta e a literatura no ponto de corte (Kneedle ou 1 desvio-padrão) é feita no caderno 5.

**Limitação**: o viés otimista do ponto de corte não é tratado por validação aninhada, por causa do custo computacional (ver as limitações do artigo).

### 1. Carregar dados e preparar X/y

In [2]:
import pandas as pd
import numpy as np
import os
import time
import json
import re
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, balanced_accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from scipy.stats import wilcoxon
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# Base de MODELAGEM: a base restante gerada pelo notebook 1 -- linhas que NÃO
# foram usadas para criar os rankings (notebook 2). Escolher os atributos e avaliar
# os modelos com as mesmas linhas inflaria os resultados (viés de seleção); por isso
# os modelos NÃO usam mais a amostra.xlsx.
arquivo_dados = "base_restante.xlsx"
pasta_base = os.getcwd()

df = pd.read_excel(os.path.join(pasta_base, arquivo_dados))
df.columns = df.columns.astype(str).str.strip()

target_column = df.columns[-1]
y = df[target_column].astype(str)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_original = df.drop(columns=target_column).select_dtypes(include=[np.number]).copy()
X_original = X_original.fillna(X_original.median())

print(f"Base de modelagem: {arquivo_dados} ({len(df)} linhas)")
print(f"Linhas por classe: {df[target_column].value_counts().to_dict()}")
print(f"Atributos numéricos disponíveis: {X_original.shape[1]}")
print(f"Classes: {list(le.classes_)}")

Base de modelagem: base_restante.xlsx (784 linhas)
Linhas por classe: {-1: 732, 1: 52}
Atributos numéricos disponíveis: 590
Classes: ['-1', '1']


### 2. Função de métricas (mesma para todos os paradigmas)

*Correção em relação ao notebook original: o AUC agora funciona tanto para classe binária quanto multiclasse (usava `y_proba[:, 1]` incondicionalmente, o que quebra sempre que a coluna de classe tem mais de 2 valores).*

In [3]:
def calcular_metricas(y_true, y_pred, y_proba, le_classes_):
    cm = confusion_matrix(y_true, y_pred, labels=le.transform(le_classes_))
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        espec = tn / (tn + fp) if (tn + fp) > 0 else 0
    else:
        tp, fp, tn, fn, sens, espec = 0, 0, 0, 0, 0, 0

    # AUC: binária usa a probabilidade da classe positiva; multiclasse usa
    # one-vs-rest com média macro (roc_auc_score exige isso para >2 classes,
    # e chamar com y_proba[:, 1] direto -- como no notebook original -- quebra
    # sempre que há mais de 2 classes).
    n_classes = y_proba.shape[1]
    if n_classes == 2:
        auc = roc_auc_score(y_true, y_proba[:, 1])
    elif n_classes > 2:
        try:
            auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
        except ValueError:
            auc = np.nan
    else:
        auc = np.nan

    return {
        "VP": tp, "FP": fp, "VN": tn, "FN": fn,
        "Sensibilidade": sens, "Especificidade": espec,
        "Acurácia": accuracy_score(y_true, y_pred),
        "Acurácia Balanceada": balanced_accuracy_score(y_true, y_pred),
        "AUC": auc,
        "Precisão (Macro Avg)": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall (Macro Avg)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1-Score (Macro Avg)": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Precisão (Weighted Avg)": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall (Weighted Avg)": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1-Score (Weighted Avg)": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

### 3. Funções genéricas de avaliação (Backward Elimination + CV)

`avaliar_subset_de_atributos` roda a validação cruzada para UM subconjunto de atributos, recebendo o modelo a treinar via `model_factory` (uma função que devolve um classificador novo a cada fold, para não vazar estado entre folds). `precisa_escalonar=True` ativa o `StandardScaler` (necessário para SVM, MLP e Regressão Logística; desnecessário — e por isso removido — para DT, RF, GB e GaussianNB, que não são sensíveis à escala dos atributos).

**Valores por fold**: além da média e do desvio de cada métrica, a linha devolvida guarda os **5 valores brutos por fold**, serializados em `{Métrica}_Por_Fold` (formato `[v1, v2, v3, v4, v5]`, lido de volta com `json.loads`). Isso permite, na seção 6 e no caderno 5, rodar testes pareados em vez de comparar só médias pontuais.

**Nova função `nome_aba_excel`**: dado o nome que o arquivo por-ranking teria antes (`Modelos<Paradigma>_CV_<Ranking>`), devolve um nome de aba válido para o Excel consolidado — até 31 caracteres, sem os caracteres proibidos `[ ] : * ? / \`. Quando dois rankings diferentes colidem no mesmo nome truncado (raro, mas possível com nomes muito longos), um sufixo numérico `~2`, `~3`... é acrescentado para manter os nomes únicos dentro do arquivo.

In [4]:
def avaliar_subset_de_atributos(atributos_usados, X, y, le_classes_, model_factory,
                                 precisa_escalonar=False, n_splits=5):
    # Cada tarefa roda num processo worker separado (joblib/loky), que não herda
    # o warnings.filterwarnings da célula 2 -- por isso é repetido aqui.
    import warnings as _w
    _w.filterwarnings("ignore")
    X_subset = X[atributos_usados]
    start_time = time.time()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    metricas_por_fold = []

    for train_index, test_index in skf.split(X_subset, y):
        X_train, X_test = X_subset.iloc[train_index], X_subset.iloc[test_index]
        y_train, y_test = y[train_index], y[test_index]

        if precisa_escalonar:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

        modelo = model_factory()
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        y_proba = modelo.predict_proba(X_test)

        metricas = calcular_metricas(y_test, y_pred, y_proba, le_classes_)
        if metricas:
            metricas_por_fold.append(metricas)

    tempo_gasto = time.time() - start_time

    if not metricas_por_fold:
        return None

    df_metricas = pd.DataFrame(metricas_por_fold)

    resultado_agregado = {
        "Nº de Atributos": len(atributos_usados),
        "Atributos Usados": ", ".join(atributos_usados),
        "Tempo de Execução (s)": round(tempo_gasto, 2),
    }

    for metrica in df_metricas.columns:
        resultado_agregado[f"{metrica} Média"] = round(df_metricas[metrica].mean(), 3)
        resultado_agregado[f"{metrica} Desvio"] = round(df_metricas[metrica].std(), 3)
        # Valores brutos por fold, para teste pareado (Wilcoxon)
        resultado_agregado[f"{metrica}_Por_Fold"] = json.dumps(
            [round(float(v), 6) for v in df_metricas[metrica].tolist()]
        )

    # Sinaliza AUC pior que aleatório sem precisar abrir a tabela inteira
    resultado_agregado["AUC Abaixo de 0.5"] = bool(resultado_agregado.get("AUC Média", np.nan) < 0.5)

    return resultado_agregado


def valores_por_fold(linha, metrica):
    '''Lê de volta a lista de 5 valores brutos gravada por avaliar_subset_de_atributos
    (usada na análise de redundância da seção 6).'''
    chave = f"{metrica}_Por_Fold"
    if chave not in linha or pd.isna(linha[chave]):
        return None
    return json.loads(linha[chave])


def teste_wilcoxon_pareado(valores_a, valores_b, alpha=0.05):
    '''Teste de Wilcoxon pareado (signed-rank) entre duas listas de mesmo tamanho
    (tipicamente os 5 valores de fold de uma métrica, em duas configurações
    diferentes -- dois subconjuntos de atributos, ou Proposto vs. Literatura).
    Devolve (p_valor, diferenca_significativa). Se as listas forem idênticas
    (todas as diferenças = 0), o Wilcoxon não é definido -- trata como "sem
    diferença" (p = 1.0) em vez de deixar o scipy levantar exceção.'''
    a = np.asarray(valores_a, dtype=float)
    b = np.asarray(valores_b, dtype=float)
    if len(a) != len(b) or len(a) < 1:
        return np.nan, False
    if np.allclose(a, b):
        return 1.0, False
    try:
        _, p_valor = wilcoxon(a, b)
    except ValueError:
        # amostra pequena demais / todas as diferenças com o mesmo sinal e zero
        return np.nan, False
    return float(p_valor), bool(p_valor <= alpha)


# Nome de aba válido para o Excel consolidado
_CARACTERES_PROIBIDOS_ABA = re.compile(r"[\[\]:*?/\\]")


def nome_aba_excel(nome_desejado, nomes_ja_usados):
    '''Sanitiza 'nome_desejado' (ex.: 'ModelosDT_CV_Ranking_ApproxMIC') para um
    nome de aba Excel válido: remove caracteres proibidos ([ ] : * ? / \\) e
    trunca para no máximo 31 caracteres. Se o resultado colidir com um nome já
    usado NESTE arquivo (nomes_ja_usados -- um set, atualizado a cada chamada),
    acrescenta um sufixo numérico curto ('~2', '~3'...) para manter unicidade.'''
    limpo = _CARACTERES_PROIBIDOS_ABA.sub("_", nome_desejado).strip()
    base = limpo[:31] if limpo else "Ranking"

    candidato = base
    contador = 2
    while candidato in nomes_ja_usados:
        sufixo = f"~{contador}"
        candidato = base[: 31 - len(sufixo)] + sufixo
        contador += 1

    nomes_ja_usados.add(candidato)
    return candidato


def rodar_modelo_cv_paralelo(caminho_ranking, model_factory, prefixo, precisa_escalonar=False,
                              n_splits=5, n_jobs=15):
    '''Lê um ranking, monta as tarefas de Backward Elimination (do maior para o menor
    subconjunto de atributos, sempre removendo o pior-ranqueado) e roda em paralelo.
    Não escreve um Excel por ranking: devolve
    (nome_ranking, df_resultados) para que o chamador acumule todos os rankings de um
    paradigma e grave um único Excel consolidado (1 aba por ranking) ao final.'''
    print(f"\n[{prefixo}] Processando em paralelo ({n_jobs} núcleos): {os.path.basename(caminho_ranking)}")
    try:
        ranking = pd.read_excel(caminho_ranking)
        ranking.columns = ranking.columns.astype(str).str.strip()
        if "Feature" not in ranking.columns:
            print("Sem coluna 'Feature'. Ignorado.")
            return None, None

        # Backward Elimination: começa com todos os atributos (na ordem do ranking,
        # melhor -> pior) e vai removendo o último (pior-ranqueado) a cada passo
        #
        # Quando TODOS os valores da coluna "Feature" de um ranking são
        # dígitos puros (ex.: nomes de sensor "0","1",...,"589", como nos
        # rankings da Proposta e das Descritivas -- diferente dos rankings da
        # Literatura, que costumam ter pelo menos um nome de atributo não
        # numérico), o pandas relê essa coluna do Excel como NÚMERO (int/float),
        # não como texto, mesmo tendo sido gravada como texto. Isso faz a
        # comparação "a in X_original.columns" falhar silenciosamente para TODO
        # o ranking (número 562 nunca é igual à string "562"), descartando o
        # ranking inteiro como "sem correspondência". Por isso cada valor é
        # normalizado para a MESMA representação textual usada nos nomes de
        # coluna de X_original antes de comparar.
        def _normalizar_feature(valor):
            if isinstance(valor, float) and valor.is_integer():
                return str(int(valor))
            return str(valor).strip()

        atributos_ordem_remocao = [
            f for f in (_normalizar_feature(a) for a in ranking["Feature"].dropna())
            if f in X_original.columns
        ]
        if not atributos_ordem_remocao:
            print("Ranking vazio ou sem correspondência com as colunas da base. Ignorado.")
            return None, None

        tarefas = [atributos_ordem_remocao[:i] for i in range(len(atributos_ordem_remocao), 0, -1)]

        resultados = Parallel(n_jobs=n_jobs, backend="loky")(
            delayed(avaliar_subset_de_atributos)(
                atributos_usados=subset,
                X=X_original,
                y=y_encoded,
                le_classes_=le.classes_,
                model_factory=model_factory,
                precisa_escalonar=precisa_escalonar,
                n_splits=n_splits,
            ) for subset in tarefas
        )

        resultados = [r for r in resultados if r is not None]
        if not resultados:
            print("❌ Nenhum resultado gerado após o processamento paralelo.")
            return None, None

        nome_ranking = os.path.splitext(os.path.basename(caminho_ranking))[0]
        df_resultados = pd.DataFrame(resultados)

        colunas_iniciais = ["Nº de Atributos", "Atributos Usados", "Tempo de Execução (s)", "AUC Abaixo de 0.5"]
        outras_colunas = [col for col in df_resultados.columns if col not in colunas_iniciais]
        df_resultados = df_resultados[colunas_iniciais + sorted(outras_colunas)]

        n_auc_baixo = int(df_resultados["AUC Abaixo de 0.5"].sum())
        if n_auc_baixo > 0:
            print(f"   {n_auc_baixo} de {len(df_resultados)} linha(s) com AUC Média < 0,50.")

        # Não grava mais Excel aqui -- devolve os dados para o chamador consolidar
        return nome_ranking, df_resultados

    except Exception as e:
        print(f"❌ Erro ao processar {os.path.basename(caminho_ranking)} com {prefixo}: {e}")
        return None, None

### 4. Os 7 paradigmas de aprendizado

`model_factory` é uma função (não uma instância) porque cada fold precisa de um modelo recém-inicializado — reaproveitar a mesma instância entre folds vazaria estado treinado. O `RandomForestClassifier` usa `n_jobs=-1` (paralelismo aninhado: cada worker do `Parallel` externo ainda abre suas próprias threads para construir as árvores), pois o RF é o paradigma mais pesado para rodar com CV de 5 folds e, na prática, isso reduz bastante o tempo total.

**Hiperparâmetros**: todos os paradigmas usam os **valores padrão do scikit-learn**, sem ajuste. Só há duas exceções, ambas obrigatórias: o SVM precisa de `probability=True` para calcular o AUC, e a Regressão Logística precisa de `l1_ratio=1.0` com `solver="liblinear"` para ser Lasso (o solver padrão não aceita L1). `random_state=42` só fixa a semente.

**Regressão Logística L1**: `LogReg_Lasso` usa penalização L1, com `C` no padrão (1,0) e sem busca de hiperparâmetro. Um grid search dentro de cada fold, de cada subconjunto e de cada ranking multiplicaria o custo computacional, o mesmo motivo pelo qual a validação aninhada não foi usada.

**Sobre a coluna `Tempo de Execução (s)`**: ela é medida dentro de cada tarefa (`avaliar_subset_de_atributos`), do início ao fim dos 5 folds daquele subconjunto de atributos -- ou seja, é o tempo real de treinar+avaliar o modelo daquele paradigma, sob as mesmas condições de paralelismo em que ele roda de fato. Junto com `Nº de Atributos` (N), essa coluna (T) é o par usado depois para `log(T)` vs `log(N)`, a regressão linear e a extração de α e do r de Pearson -- os dois já saem prontos em toda linha/aba do `Modelos<Paradigma>_CV_Consolidado.xlsx`, então essa etapa de escalabilidade pode consumir esse Excel diretamente, sem precisar rodar os modelos de novo.

In [5]:
# Todos os paradigmas com os HIPERPARÂMETROS PADRÃO do scikit-learn. Só são passados
# argumentos que não mudam o aprendizado ou que são obrigatórios:
#   - random_state=42: só fixa a semente (reprodutibilidade), não é hiperparâmetro;
#   - n_jobs=-1 no RF: só paraleliza, não muda o modelo;
#   - SVC(probability=True): OBRIGATÓRIO -- sem ele não há predict_proba e o AUC dá erro;
#   - Regressão Logística Lasso: o padrão é L2; para ser L1 é OBRIGATÓRIO l1_ratio=1.0
#     e um solver que aceite L1 (o padrão, lbfgs, dá erro). A forma antiga penalty="l1"
#     foi descontinuada no scikit-learn 1.8 e deixa de existir no 1.10.
#     C=1.0 e max_iter=100 são os próprios valores padrão, deixados explícitos.
C_LASSO = 1.0  # só usado pelo LogReg_Lasso (1.0 é o padrão do scikit-learn)

PARADIGMAS = {
    "DT":  {"model_factory": lambda: DecisionTreeClassifier(random_state=42),
            "precisa_escalonar": False},
    "RF":  {"model_factory": lambda: RandomForestClassifier(random_state=42, n_jobs=-1),
            "precisa_escalonar": False},
    "GB":  {"model_factory": lambda: GradientBoostingClassifier(random_state=42),
            "precisa_escalonar": False},
    "SVM": {"model_factory": lambda: SVC(random_state=42, probability=True),  # probability=True exigido p/ predict_proba (AUC) -- não é tuning
            "precisa_escalonar": True},
    "MLP": {"model_factory": lambda: MLPClassifier(random_state=42),
            "precisa_escalonar": True},
    "GNB": {"model_factory": lambda: GaussianNB(),
            "precisa_escalonar": False},
    # Regressão Logística com regularização L1 (Lasso)
    "LogReg_Lasso": {"model_factory": lambda: LogisticRegression(
                          l1_ratio=1.0, solver="liblinear", C=C_LASSO,
                          random_state=42, max_iter=100),
                      "precisa_escalonar": True},
}

print("Paradigmas configurados:", list(PARADIGMAS.keys()))


Paradigmas configurados: ['DT', 'RF', 'GB', 'SVM', 'MLP', 'GNB', 'LogReg_Lasso']


### 5. Processamento principal

Roda cada um dos 7 paradigmas contra **todos** os arquivos `Ranking_*.xlsx` encontrados na pasta. Para cada paradigma, acumula os resultados de todos os rankings de um paradigma num dicionário e, ao terminar esse paradigma, grava **um único arquivo** `Modelos<Paradigma>_CV_Consolidado.xlsx` com **uma aba por ranking**.

**Sobre o nome da aba**: usar o nome completo (`Modelos<Paradigma>_CV_<Ranking>`, ex.: `ModelosDT_CV_Ranking_ApproxMIC`) colide com o limite de 31 caracteres do Excel: em paradigmas de nome longo (ex.: `LogReg_Lasso`), o prefixo consome boa parte dos 31 caracteres. Como o paradigma já está no nome do arquivo `.xlsx`, a aba usa **só o nome do ranking** (ex.: `Ranking_ApproxMIC`), que fica legível e identifica o ranking sem ambiguidade. O nome completo continua registrado, ao lado do nome da aba, no arquivo abaixo.

É salvo `Mapa_Abas_Consolidado.xlsx`: para cada paradigma, de-para entre o nome da aba gravada (`Nome da Aba`), o nome completo do ranking (`Ranking`) e o nome que o arquivo individual teria antes da consolidação (`Nome Arquivo Antigo`) -- útil caso algum nome de aba tenha sido truncado (ex.: duas descritivas com `_BaseCompleta` cujos 31 primeiros caracteres coincidem) e seja preciso confirmar de qual ranking ele veio.

**Atenção ao tempo**: o número de execuções é `nº de rankings × nº de paradigmas × nº de atributos` (cada um com 5 folds). Com muitos rankings e atributos, isso pode demorar bastante — ajuste `n_jobs` conforme os núcleos disponíveis, ou rode um paradigma de cada vez comentando os outros no dicionário `PARADIGMAS` acima.

In [6]:
try:
    arquivos_ranking = sorted([
        os.path.join(pasta_base, f) for f in os.listdir(pasta_base)
        if f.startswith("Ranking_") and f.endswith(".xlsx")
    ])
    if not arquivos_ranking:
        print("Nenhum arquivo 'Ranking_*.xlsx' encontrado em", pasta_base)
except Exception as e:
    print(f"❌ Erro ao listar arquivos de ranking: {e}")
    arquivos_ranking = []

print(f"{len(arquivos_ranking)} ranking(s) encontrado(s) para {len(PARADIGMAS)} paradigma(s).")

resumos_por_paradigma = {prefixo: [] for prefixo in PARADIGMAS}
auc_baixo_geral = []  # consolida em todos os paradigmas/rankings
mapa_abas_geral = []  # de-para aba <-> ranking <-> nome de arquivo antigo, para todos os paradigmas

for prefixo, config in PARADIGMAS.items():
    # Acumula {nome_aba: df_resultados} deste paradigma -- vira as abas do Excel consolidado ao final do loop de rankings
    abas_deste_paradigma = {}
    nomes_abas_usados = set()

    for ranking_path in arquivos_ranking:
        start_geral = time.time()
        nome_ranking, df_resultados = rodar_modelo_cv_paralelo(
            ranking_path,
            model_factory=config["model_factory"],
            prefixo=prefixo,
            precisa_escalonar=config["precisa_escalonar"],
            n_splits=5,
            n_jobs=12,
        )
        end_geral = time.time()
        print(f"--- [{prefixo}] Tempo total para '{os.path.basename(ranking_path)}': {end_geral - start_geral:.2f}s ---")

        if df_resultados is None:
            continue

        # Nome que teria o arquivo individual por ranking, registrado no Mapa_Abas_Consolidado.xlsx.
        nome_arquivo_antigo = f"Modelos{prefixo}_CV_{nome_ranking}"
        # A aba usa só o nome do ranking (o paradigma já está no nome do arquivo .xlsx);
        # o nome completo fica no Mapa_Abas_Consolidado.xlsx, por causa do limite de 31 caracteres do Excel.
        nome_aba = nome_aba_excel(nome_ranking, nomes_abas_usados)
        abas_deste_paradigma[nome_aba] = df_resultados

        mapa_abas_geral.append({
            "Paradigma": prefixo,
            "Ranking": nome_ranking,
            "Nome Arquivo Antigo": nome_arquivo_antigo,
            "Nome da Aba": nome_aba,
        })

        coluna_principal = "Acurácia Balanceada Média"
        if coluna_principal in df_resultados.columns and not df_resultados.empty:
            idx_melhor = df_resultados[coluna_principal].idxmax()
            melhor = df_resultados.loc[idx_melhor]
            resumos_por_paradigma[prefixo].append({"Ranking": nome_ranking, **melhor.to_dict()})

        # Coleta linhas com AUC Média < 0.5 para o resumo consolidado
        if "AUC Abaixo de 0.5" in df_resultados.columns:
            linhas_baixo = df_resultados[df_resultados["AUC Abaixo de 0.5"]]
            for _, linha in linhas_baixo.iterrows():
                auc_baixo_geral.append({
                    "Paradigma": prefixo, "Ranking": nome_ranking,
                    "Nº de Atributos": linha["Nº de Atributos"],
                    "AUC Média": linha.get("AUC Média", np.nan),
                })

    # Grava o Excel consolidado deste paradigma -- 1 aba por ranking
    if abas_deste_paradigma:
        caminho_consolidado = os.path.join(pasta_base, f"Modelos{prefixo}_CV_Consolidado.xlsx")
        with pd.ExcelWriter(caminho_consolidado, engine="openpyxl") as writer:
            for nome_aba, df_aba in abas_deste_paradigma.items():
                df_aba.to_excel(writer, sheet_name=nome_aba, index=False)
        print(f"\n[{prefixo}] Consolidado salvo: {caminho_consolidado} "
              f"({len(abas_deste_paradigma)} aba(s)/ranking(s))")
    else:
        print(f"\n[{prefixo}] Nenhum resultado gerado -- consolidado não foi criado.")

# Salvar um resumo geral por paradigma
for prefixo, resumo_geral in resumos_por_paradigma.items():
    if resumo_geral:
        df_resumo = pd.DataFrame(resumo_geral)
        resumo_path = os.path.join(pasta_base, f"Resumo_Geral_Modelos{prefixo}_CV.xlsx")
        df_resumo.to_excel(resumo_path, index=False)
        print(f"[{prefixo}] Resumo final salvo como: {resumo_path}")
    else:
        print(f"[{prefixo}] Nenhum resultado foi gerado para o resumo final.")

# Mapa de-para aba <-> ranking <-> nome de arquivo antigo (todos os paradigmas)
if mapa_abas_geral:
    df_mapa_abas = pd.DataFrame(mapa_abas_geral)
    path_mapa_abas = os.path.join(pasta_base, "Mapa_Abas_Consolidado.xlsx")
    df_mapa_abas.to_excel(path_mapa_abas, index=False)
    print(f"\n Mapa aba <-> ranking salvo em: {path_mapa_abas}")

# Resumo consolidado de AUC abaixo de 0.5, em todos os paradigmas/rankings
if auc_baixo_geral:
    df_auc_baixo = pd.DataFrame(auc_baixo_geral).sort_values("AUC Média")
    path_auc_baixo = os.path.join(pasta_base, "Alerta_AUC_Abaixo_0.5.xlsx")
    df_auc_baixo.to_excel(path_auc_baixo, index=False)
    print(f"\n{len(df_auc_baixo)} combinação(ões) paradigma/ranking com AUC Média < 0,50 "
          f"-- ver {path_auc_baixo}")
else:
    print("\nNenhuma combinação com AUC Média abaixo de 0,50.")

📋 43 ranking(s) encontrado(s) para 7 paradigma(s).

🚀 [DT] Processando em paralelo (12 núcleos): Ranking_ANOVA.xlsx
   ⚠️ 59 de 590 linha(s) com AUC Média < 0,50 (item 11, orientador).
--- [DT] Tempo total para 'Ranking_ANOVA.xlsx': 83.62s ---

🚀 [DT] Processando em paralelo (12 núcleos): Ranking_AmplitudeSigma.xlsx
   ⚠️ 50 de 590 linha(s) com AUC Média < 0,50 (item 11, orientador).
--- [DT] Tempo total para 'Ranking_AmplitudeSigma.xlsx': 86.43s ---

🚀 [DT] Processando em paralelo (12 núcleos): Ranking_AmplitudeSigma_BaseCompleta.xlsx
   ⚠️ 45 de 590 linha(s) com AUC Média < 0,50 (item 11, orientador).
--- [DT] Tempo total para 'Ranking_AmplitudeSigma_BaseCompleta.xlsx': 103.97s ---

🚀 [DT] Processando em paralelo (12 núcleos): Ranking_ApproxMIC.xlsx
   ⚠️ 43 de 590 linha(s) com AUC Média < 0,50 (item 11, orientador).
--- [DT] Tempo total para 'Ranking_ApproxMIC.xlsx': 114.22s ---

🚀 [DT] Processando em paralelo (12 núcleos): Ranking_Assimetria.xlsx
   ⚠️ 44 de 590 linha(s) com AUC Mé

### 6. Redundância de variáveis

Pergunta: se uma variável é retirada e a acurácia balanceada não cai, ela pode ser considerada redundante?

Esta seção lê os arquivos `Modelos<Paradigma>_CV_Consolidado.xlsx` (um por paradigma, uma aba por ranking). Para cada aba (ranking) de cada paradigma, percorre os passos consecutivos do Backward Elimination (N atributos → N-1 atributos) e roda um **teste de Wilcoxon pareado** entre os 5 valores de fold de Acurácia Balanceada de cada passo (usando `teste_wilcoxon_pareado`, definida na seção 3). Se o teste não encontrar diferença significativa (p > 0,05) — ou se a acurácia até melhorar com a remoção — a variável removida naquele passo é marcada como **redundante** para aquela combinação (ranking, paradigma).

Gera dois arquivos:
- `Redundancia_Por_Combinacao.xlsx`: uma linha por passo de remoção (paradigma, ranking, atributo removido, p-valor, se é redundante).
- `Redundancia_Resumo_Por_Variavel.xlsx`: quantas vezes cada variável foi marcada redundante, e em qual fração das combinações — uma variável redundante em quase todas as combinações é um candidato forte a ser descartada de forma mais ampla, uma que só é redundante em uma combinação isolada provavelmente não é.

In [7]:
arquivos_consolidados = sorted([
    f for f in os.listdir(pasta_base)
    if f.startswith("Modelos") and f.endswith("_CV_Consolidado.xlsx")
])
print(f"{len(arquivos_consolidados)} arquivo(s) 'Modelos<Paradigma>_CV_Consolidado.xlsx' "
      f"encontrado(s) para avaliar redundância.")

METRICA_REDUNDANCIA = "Acurácia Balanceada"  # métrica usada para decidir redundância
registros_redundancia = []

for arquivo in arquivos_consolidados:
    # nome do arquivo: Modelos<Paradigma>_CV_Consolidado.xlsx
    m = re.match(r"^Modelos(.+?)_CV_Consolidado\.xlsx$", arquivo)
    if not m:
        print(f"Nome fora do padrão, ignorado: {arquivo}")
        continue
    paradigma = m.group(1)

    # sheet_name=None lê TODAS as abas de uma vez, devolvendo {nome_aba: DataFrame}
    todas_as_abas = pd.read_excel(os.path.join(pasta_base, arquivo), sheet_name=None)

    for nome_aba, df_modelo in todas_as_abas.items():
        # O nome da aba já É o nome do ranking (seção 5 nomeia a aba só pelo
        # ranking, sem repetir "Modelos<Paradigma>_CV_") -- usado direto aqui.
        nome_ranking_exibicao = nome_aba
        if f"{METRICA_REDUNDANCIA}_Por_Fold" not in df_modelo.columns:
            print(f"Aba '{nome_aba}' de '{arquivo}' não tem colunas por fold "
                  f"(gerada com versão antiga do notebook?). Ignorada.")
            continue

        # Ordena do maior N (todos os atributos) para o menor -- ordem em que o
        # Backward Elimination de fato removeu variáveis
        df_modelo = df_modelo.sort_values("Nº de Atributos", ascending=False).reset_index(drop=True)

        for i in range(len(df_modelo) - 1):
            linha_antes = df_modelo.iloc[i]       # N atributos
            linha_depois = df_modelo.iloc[i + 1]  # N-1 atributos (uma variável a menos)

            atributos_antes = set(a.strip() for a in linha_antes["Atributos Usados"].split(","))
            atributos_depois = set(a.strip() for a in linha_depois["Atributos Usados"].split(","))
            removidos = atributos_antes - atributos_depois
            if len(removidos) != 1:
                # nomes de atributo com vírgula quebrariam esse split -- não deveria
                # acontecer com os nomes curtos usados no pipeline, mas fica o aviso
                print(f"Esperava 1 atributo removido entre N={linha_antes['Nº de Atributos']} e "
                      f"N={linha_depois['Nº de Atributos']} na aba '{nome_aba}' de {arquivo}, "
                      f"encontrei {len(removidos)}. Ignorado.")
                continue
            atributo_removido = removidos.pop()

            folds_antes = valores_por_fold(linha_antes, METRICA_REDUNDANCIA)
            folds_depois = valores_por_fold(linha_depois, METRICA_REDUNDANCIA)
            p_valor, diferenca_sig = teste_wilcoxon_pareado(folds_antes, folds_depois)

            acc_antes = linha_antes[f"{METRICA_REDUNDANCIA} Média"]
            acc_depois = linha_depois[f"{METRICA_REDUNDANCIA} Média"]
            nao_piorou = acc_depois >= acc_antes
            redundante = nao_piorou and not diferenca_sig

            registros_redundancia.append({
                "Paradigma": paradigma,
                "Ranking": nome_ranking_exibicao,
                "Atributo Removido": atributo_removido,
                "Nº Atributos Antes": linha_antes["Nº de Atributos"],
                "Nº Atributos Depois": linha_depois["Nº de Atributos"],
                f"{METRICA_REDUNDANCIA} Antes": acc_antes,
                f"{METRICA_REDUNDANCIA} Depois": acc_depois,
                "p-valor (Wilcoxon)": round(p_valor, 4) if not np.isnan(p_valor) else np.nan,
                "Redundante": redundante,
            })

if registros_redundancia:
    df_redundancia = pd.DataFrame(registros_redundancia)
    path_red = os.path.join(pasta_base, "Redundancia_Por_Combinacao.xlsx")
    df_redundancia.to_excel(path_red, index=False)
    print(f"{df_redundancia['Redundante'].sum()} de {len(df_redundancia)} remoções marcadas como redundantes.")
    print(f"Salvo: {path_red}")

    df_resumo_var = (
        df_redundancia.groupby("Atributo Removido")
        .agg(
            Vezes_Avaliada=("Redundante", "size"),
            Vezes_Redundante=("Redundante", "sum"),
        )
        .reset_index()
    )
    df_resumo_var["Fração Redundante"] = (df_resumo_var["Vezes_Redundante"] / df_resumo_var["Vezes_Avaliada"]).round(3)
    df_resumo_var = df_resumo_var.sort_values("Fração Redundante", ascending=False).reset_index(drop=True)

    path_resumo_var = os.path.join(pasta_base, "Redundancia_Resumo_Por_Variavel.xlsx")
    df_resumo_var.to_excel(path_resumo_var, index=False)
    print(f"Salvo: {path_resumo_var}")
    print(df_resumo_var.head(10).to_string(index=False))
else:
    print("Nenhum arquivo de modelo válido encontrado para avaliar redundância.")

📋 7 arquivo(s) 'Modelos<Paradigma>_CV_Consolidado.xlsx' encontrado(s) para avaliar redundância.
✅ 126425 de 177289 remoções marcadas como redundantes.
✅ Salvo: c:\Users\USUARIO\Desktop\doutorado\Códigos para o Artigo\Redundancia_Por_Combinacao.xlsx
✅ Salvo: c:\Users\USUARIO\Desktop\doutorado\Códigos para o Artigo\Redundancia_Resumo_Por_Variavel.xlsx
Atributo Removido  Vezes_Avaliada  Vezes_Redundante  Fração Redundante
              266             301               251              0.834
              579             301               249              0.827
              228             301               249              0.827
              528             301               248              0.824
              240             301               247              0.821
              322             301               247              0.821
              403             301               247              0.821
               93             301               245              0.814
         